The original notebook was run in goggle collab with t4 

In [ ]:
from pathlib import Path
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from huggingface_hub import hf_hub_download
import json, torch
from scipy.sparse import csr_matrix
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_fscore_support
from google.colab import userdata
from huggingface_hub import HfApi
from pathlib import Path


In [ ]:
class AspectAttention(nn.Module):
    def __init__(self, hidden_dim, aspect_emb_dim, d_k=256, d_v=256):
        super().__init__()
        self.Wq = nn.Linear(aspect_emb_dim, d_k, bias=False)
        self.Wk = nn.Linear(hidden_dim, d_k, bias=False)
        self.Wv = nn.Linear(hidden_dim, d_v, bias=False)
        self.Wo = nn.Linear(d_v, hidden_dim)

    def forward(self, enc, aspect_emb, attention_mask):
        """
        enc: (B, L, H)
        aspect_emb: (B, N, E)  <-- Now expects 3D input (Standard)
        """
        Q = self.Wq(aspect_emb)          # (B, N, d_k)
        K = self.Wk(enc)                 # (B, L, d_k)
        V = self.Wv(enc)                 # (B, L, d_v)

        scores = torch.matmul(Q, K.transpose(1, 2)) / (Q.size(-1) ** 0.5)

        mask = attention_mask.squeeze(-1).unsqueeze(1) # (B, 1, L)
        scores = scores.masked_fill(mask == 0, -1e4)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)  # (B, N, d_v)

        return self.Wo(context)   #(B, N, H)


In [ ]:
def visualize_label_distinctness(model, label_names=None):
    """
    Plots the Cosine Similarity Matrix of the model's learned query vectors.
    Args:
        model: Your PyTorch model (must have model.head_query)
    """
    # Shape becomes (Num_Labels, Hidden_Dim)
    with torch.no_grad():
        q = model.head_query.detach().cpu()
        if q.dim() == 3:
            q = q.squeeze(0)

    q_norm = F.normalize(q, p=2, dim=1)

    similarity_matrix = torch.mm(q_norm, q_norm.t()).numpy()

    if label_names is None:
        label_names = [f"L{i}" for i in range(len(similarity_matrix))]

    avg_off_diag = (np.sum(np.abs(similarity_matrix)) - len(similarity_matrix)) / (len(similarity_matrix)**2 - len(similarity_matrix))
    print(f"Average Off-Diagonal Similarity: {avg_off_diag:.4f}")
    if avg_off_diag > 0.5:
        print("WARNING: High overlap detected. Your aspect heads are collapsing!")
    else:
        print("SUCCESS: Vectors are distinct.")


In [ ]:
def load_tokenizer_and_model_from_hf(cfg, top_dim, sub_dim, device):
    hf_repo_id = cfg.hf_repo
    hf_subfolder = cfg.hf_encoder_subfolder

    print(f"🔹 Loading Tokenizer & Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    try:
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id, subfolder=hf_subfolder)
        print(f"Tokenizer loaded.")
    except Exception as e:
        print(f"Failed to load tokenizer: {e}")
        tokenizer = AutoTokenizer.from_pretrained(hf_repo_id)

    print(f" Initializing Base Architecture from: {hf_repo_id} (subfolder: {hf_subfolder})")

    # Initialize JointModel with YOUR domain encoder as the base
    model = HierarchicalClassifier(hf_repo_id,
            subfolder=hf_subfolder,
            top_dim=top_dim,
            sub_dim=sub_dim,
            dropout=0.2).to(device)

    return tokenizer, model

In [ ]:
def class_wise_eval(y_true, y_pred_probs):
    n_classes = y_true.shape[1]
    print("-" * 85)
    print(f"{'Class ID':<10} | {'Threshold':<10} | {'F1':<10} | {'Precision':<10} | {'Recall':<10} | {'Support':<10}")
    print("-" * 85)
    final_avg_thresholds = np.array([0.5]*n_classes)
    class_metrics = {}
    macro_f1_scores = []

    for c in range(n_classes):
        preds_bin = (y_pred_probs[:, c] > final_avg_thresholds[c]).astype(int)
        true_bin = y_true[:, c]

        precision = precision_score(true_bin, preds_bin, zero_division=0)
        recall = recall_score(true_bin, preds_bin, zero_division=0)
        f1 = f1_score(true_bin, preds_bin, zero_division=0)
        support = true_bin.sum() * 9

        macro_f1_scores.append(f1)

        class_metrics[c] = {
            "threshold": final_avg_thresholds[c],
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "support": support
        }

        print(f"{c} | {final_avg_thresholds[c]:.3f}      | {f1:.4f}     | {precision:.4f}     | {recall:.4f}     | {int(support)}")

    print("-" * 85)
    avg_f1 = np.mean(macro_f1_scores)
    print(f"Final Macro F1 Score: {avg_f1:.4f}")

    return final_avg_thresholds, avg_f1, class_metrics

In [ ]:
class Config:
    output_dir = Path(r"/content/stage2_joint")  # Hugging Face repo or local path
    hf_repo = "Faisal191/aspect-classifier"  
    hf_encoder_subfolder = "Domain_trained_encoder" 
    data_path = Path(r"/content/final_aspa_data_hierarchical_with_sentiments_temp_v6.json")
    epochs = 15
    batch_size = 16
    lr_encoder = 1e-5
    lr_heads = 1e-4
    max_len = 256
    val_split = 0.1
    use_amp = True
    freeze_encoder_epochs = 0
    unfreeze_last_layers_epoch = 0
    unfreeze_full_epoch = 0
    use_sampler = False
    top_k_train = 3
    top_k_eval = 3
    prob_transfer_weight = 0.4
    gating_mode = "mul"
    clip_grad_norm = 1.0
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42
    USE_HIER_MASK_IN_EVAL = True 

cfg = Config()
cfg.output_dir.mkdir(parents=True, exist_ok=True)


torch.manual_seed(cfg.seed)
np.random.seed(cfg.seed)
torch.backends.cudnn.benchmark = True
DEVICE = cfg.device
print("Device:", DEVICE)

In [ ]:
def csr_to_torch_sparse(csr):
    coo = csr.tocoo()
    if coo.nnz == 0:
        indices = torch.empty((2,0), dtype=torch.long)
        values = torch.empty((0,), dtype=torch.float32)
    else:
        indices = torch.tensor(np.array([coo.row, coo.col]), dtype=torch.long)
        values = torch.tensor(coo.data, dtype=torch.float32)
    return torch.sparse_coo_tensor(indices, values, torch.Size(coo.shape))


def compute_metrics_numpy(y_true, y_pred, threshold=0.5):
    y_pred_bin = (y_pred > threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred_bin, average="macro", zero_division=0)
    return precision, recall, f1

In [ ]:
class HierarchicalDataset(torch.utils.data.Dataset):
    def __init__(self, data, tokenizer, max_len=256):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = item.get("text","")
        enc = self.tokenizer(text, truncation=True, padding="max_length",
                             max_length=self.max_len, return_tensors="pt")
        enc = {k:v.squeeze(0) for k,v in enc.items()}
        top_ids = np.array(item["top_cluster_ids"], dtype=np.float32)
        sub_ids = np.array(item["sub_cluster_ids"], dtype=np.float32)
        sub_sparse = csr_to_torch_sparse(csr_matrix(sub_ids.reshape(1,-1)))
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "top_labels": torch.tensor(top_ids, dtype=torch.float32),
            "sub_labels_sparse": sub_sparse
        }

def collate_fn(batch):
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "top_labels": torch.stack([b["top_labels"] for b in batch]),
        "sub_labels_sparse": [b["sub_labels_sparse"] for b in batch]
    }


In [ ]:
class HierarchicalClassifier(nn.Module):
    def __init__(self, encoder_name, top_dim, sub_dim, dropout=0.2, subfolder=None):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name, subfolder=subfolder) # Pass subfolder
        self.thresholds = torch.tensor([0.5]*top_dim, dtype=torch.float32)
        h_dim = self.encoder.config.hidden_size
        self.top_dim = top_dim
        self.aspect_emb_dim = h_dim
        self.head_aspect_attention = AspectAttention(
            hidden_dim=h_dim, aspect_emb_dim=self.aspect_emb_dim)
        self.head_query = nn.Parameter(torch.randn(1, top_dim, h_dim))
        self.head_query = nn.Parameter(torch.empty(1, top_dim, h_dim))
        nn.init.normal_(self.head_query, mean=0, std=0.02)
        self.head_gate_proj = nn.Linear(h_dim * 2, h_dim * 2)
        # Stage 1 (top-level)

        self.head_norm = nn.LayerNorm(h_dim)
        self.head_weight = nn.Parameter(torch.randn(top_dim, h_dim))
        self.head_bias = nn.Parameter(torch.zeros(top_dim))

        # Initialization
        torch.nn.init.normal_(self.head_weight, std=0.02)
        self.head_dropout = nn.Dropout(dropout)

        # Mapping-related attributes
        self.top_to_sub_map = None          # sparse tensor (top_dim × sub_dim)
        self.top_to_sub_dense = None        # dense cache for matmul
        self.top_dim = top_dim
        self.sub_dim = sub_dim

    def set_top_to_sub_map(self, mapping: dict):
        self.top_to_sub_dict = mapping 
        rows, cols = [], []
        for t, subs in mapping.items():
            rows.extend([t] * len(subs))
            cols.extend(subs)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values = torch.ones(len(rows), dtype=torch.float32)
        sparse_map = torch.sparse_coo_tensor(indices, values, (self.top_dim, self.sub_dim))
        self.top_to_sub_map = sparse_map.coalesce()
        self.top_to_sub_dense = None


    def _get_dense_map(self, device, dtype):
        """
        Safely gets (or builds) dense mapping tensor.
        - Auto moves to correct device/dtype (handles AMP).
        - Caches result for reuse.
        """
        if self.top_to_sub_dense is None or self.top_to_sub_dense.device != device:
            dense_map = self.top_to_sub_map.to_dense().to(device)
            self.top_to_sub_dense = dense_map
        if self.top_to_sub_dense.dtype != dtype:
            self.top_to_sub_dense = self.top_to_sub_dense.to(dtype)
        return self.top_to_sub_dense
    
    def forward(self, input_ids, attention_mask, gating_mode="add", prob_weight=0.5):

        enc = self.encoder(input_ids, attention_mask=attention_mask).last_hidden_state
        batch_size = enc.size(0)
        mask = attention_mask.unsqueeze(-1).float() #(B, L, 1)
        pooled = (enc * mask).sum(1) / mask.sum(1).clamp(min=1.0)
        top_query_expanded = self.head_query.expand(batch_size, -1, -1) # (B, N, H)
        aspect_context = self.head_aspect_attention(enc, top_query_expanded, mask) # (B, N, H)
        pooled_expanded = pooled.unsqueeze(1).expand(-1, self.top_dim, -1) # (B, N, H)
        # Concatenate: (B, N, 2*H)
        combined = torch.cat([pooled_expanded, aspect_context], dim=2)
        # Compute Gate Alpha: (B, N, H)
        alpha = torch.sigmoid(self.head_gate_proj(combined))
        alpha_pooled, alpha_context = alpha.chunk(2, dim=-1)

        fused_embedding = alpha_pooled * pooled_expanded + alpha_context * aspect_context
        fused_norm = self.head_norm(self.head_dropout(fused_embedding))

        top_logits = (fused_norm * self.head_weight).sum(dim=-1) + self.head_bias

        return top_logits

    def compute_loss(self, top_logits, y_top):
        device = top_logits.device
        y_top = y_top.to(dtype=torch.float32, device=device)
        top_loss = F.binary_cross_entropy_with_logits(top_logits, y_top, pos_weight=self.top_pos_weight.to(device) if self.top_pos_weight is not None else None)

        total_loss = top_loss
        top_loss = top_loss.detach()
        return total_loss

In [ ]:
def evaluate(model, dataloader, device, epoch, top_thresholds=None):
    model.eval()
    all_y_top, all_p_top = [], []
    all_p_top_cont = []
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=f"Eval E{epoch}"):
            ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
            y_top = batch["top_labels"].cpu().numpy()
            top_logits = model(ids, mask)
            p_top = torch.sigmoid(top_logits).cpu().numpy()

            if top_thresholds is not None:
                p_top_bin = (p_top > np.array(top_thresholds)).astype(int)
            else:
                p_top_bin = (p_top > 0.5).astype(int)

            all_y_top.append(y_top); all_p_top.append(p_top_bin)
            all_p_top_cont.append(p_top)

    y_top = np.vstack(all_y_top); p_top = np.vstack(all_p_top)
    p_top_cont = np.vstack(all_p_top_cont)
    class_wise_eval(y_top, p_top_cont)

    p_top_th, r_top_th, f1_top = compute_metrics_numpy(y_top, p_top)

    return {"top_f1":f1_top,
            "p_top":p_top_th,"r_top":r_top_th,
             "y_top": y_top, "p_top_cont": p_top_cont}


In [ ]:
def train(cfg):
    data = json.load(open(cfg.data_path))
    top_to_sub_map = defaultdict(set)
    for it in data:
        for k,v in it.get("top_to_sub_ids", {}).items():
            top_to_sub_map[int(k)].update(v)
    top_to_sub_map = {k:list(v) for k,v in top_to_sub_map.items()}

    TOP, SUB = len(data[0]["top_cluster_ids"]), len(data[0]["sub_cluster_ids"])

    tokenizer, model = load_tokenizer_and_model_from_hf(cfg, TOP, SUB, DEVICE)

    dataset = HierarchicalDataset(data, tokenizer, max_len=cfg.max_len)
    val_size = int(cfg.val_split*len(dataset))
    train_data, val_data = random_split(dataset, [len(dataset)-val_size, val_size])
    np.save(cfg.output_dir/"valid_indices.npy", np.array(val_data.indices))

    train_loader = DataLoader(train_data, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_data, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate_fn)

    model.set_top_to_sub_map(top_to_sub_map)

    # Compute label weights
    top_counts = np.sum([it["top_cluster_ids"] for it in data], axis=0)
    model.top_pos_weight = torch.tensor((len(data)-top_counts)/(top_counts+1e-6),dtype=torch.float32,device=DEVICE)

    encoder_params = []
    new_head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "encoder" in name:
            encoder_params.append(param)
        else:
            new_head_params.append(param)

    optimizer = torch.optim.AdamW([
        {"params": encoder_params,   "lr": cfg.lr_encoder}, # Slower learning for base
        {"params": new_head_params,  "lr": cfg.lr_heads}    # Faster learning for new layers
    ], weight_decay=0.01)

    total_steps = len(train_loader)*cfg.epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)
    scaler = torch.amp.GradScaler(enabled=cfg.use_amp)

    start_epoch, best_sum = 1, 0.0

    for epoch in range(start_epoch, cfg.epochs+1):
        if epoch == cfg.unfreeze_last_layers_epoch:
            for name,p in model.encoder.named_parameters():
                if "block.6" in name or "block.7" in name: p.requires_grad=True
            print(f"Unfroze last layers at epoch {epoch}.")
        if epoch == cfg.unfreeze_full_epoch:
            for p in model.encoder.parameters(): p.requires_grad=True
            print(f"Unfroze full encoder at epoch {epoch}.")

        model.train()
        for batch in tqdm(train_loader, desc=f"Train E{epoch}"):
            ids, mask = batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)
            y_top = batch["top_labels"].to(DEVICE)

            with torch.amp.autocast(device_type='cuda', enabled=cfg.use_amp):
                top_logits = model(ids, mask, cfg.gating_mode, cfg.prob_transfer_weight)
                loss = model.compute_loss(top_logits, y_top)

            if not torch.isfinite(loss):
                print(f" Skipping batch due to non-finite loss at loss {loss}")
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_grad_norm)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()

        # NaN-safe threshold clipping
        sub_thresholds = np.nan_to_num(sub_thresholds, nan=0.45, posinf=0.5, neginf=0.4)
        sub_thresholds = np.clip(sub_thresholds, 0.35, 0.6)

        # Eval
        print(f"Epoch {epoch}")
        metrics = evaluate(model, val_loader, DEVICE, epoch, top_thresholds=[0.5]*TOP)

        # Save best by F1 sum
        f1_sum = metrics["top_f1"]
        if f1_sum > best_sum:
            best_sum = f1_sum
            torch.save(model.state_dict(), cfg.output_dir/"best_stage1_v6.pt")
            print(" Saved new best model (F1 sum improved).")
        visualize_label_distinctness(model)

    print(f" Training finished. Best combined F1: {best_sum:.4f}")

if __name__=="__main__":
    train(cfg)

In [ ]:
hf_token = userdata.get('HF_TOKEN')

# Initialize HfApi with the token
api = HfApi(token=hf_token)

username = "Faisal191"
repo_name = "aspect-classifier"
repo_id = f"{username}/{repo_name}"

checkpoint_path = Path(r"/content/stage2_joint/best_stage1_v6.pt")

# Attempt the upload again with proper authentication
api.upload_file(
    path_or_fileobj=checkpoint_path,
    path_in_repo="final_stage1_v6/best_stage1.pt",
    repo_id=repo_id,
    commit_message="Upload final checkpoint")
